# Sentimiento y tópicos en reseñas turísticas de Cundinamarca
**Procesamiento de Lenguaje Natural · Grupo 901 · Universidad de Cundinamarca**
Docente: Ing. Freddy Orlando Martínez Bernal
Integrantes: Brandon I. Castiblanco, Juan M. Pérez, Kevin L. Ramírez, Luigy C. Arias

En este cuaderno hacemos todo el análisis, de arriba hacia abajo:

| Parte | Qué hacemos | Qué sale |
|---|---|---|
| 0 | Instalar y configurar | parámetros en `CONFIG` |
| 1 | Registro Nacional de Turismo (RNT) | `rnt_limpio.csv` |
| 2 | Armar el corpus desde los CSV del scraper, cargar y limpiar | `resenas_limpias.csv` |
| 3 | Preparar el texto (spaCy) | tokens por reseña |
| 4 | Sentimiento de −1 a +1, **por oración**, y validación (automática y manual) | `resenas_procesadas.csv`, `muestra_validacion_manual.csv` |
| 5 | TF-IDF y alertas en oraciones negativas | `tfidf_*.csv`, `alertas_*.csv` |
| 6 | Tópicos con LDA | `topicos_*.csv` |
| 6b | Temas de problema (vías, precios, atención...) y reglas de decisión | `problemas_por_tema.csv`, `reglas_decision.csv`, `oraciones_negativas.csv` |
| 7 | Cruce con el RNT | `metricas_municipio.csv` |
| 8 | Exportar todo | carpeta `salidas/` y un `.zip` |

**Sobre los datos:** las reseñas son reales, sacadas de Google Maps lugar por lugar (parques, plazas, miradores y atractivos de cada municipio) con Instant Data Scraper. Basta con subir el `.zip` con los CSV tal como salieron del scraper (nombrados `<municipio>_<lugar>.csv`); el cuaderno los consolida solo. No inventa reseñas: si no encuentra datos, se detiene.

**Qué hay que tener presente al leer los resultados**
- Las reseñas turísticas de Google son muy positivas por naturaleza y los textos son cortos (mediana de ~125 caracteres). Por eso el sentimiento promedio sale alto y el valor está en **las críticas puntuales**: una reseña puede ser positiva en general y aun así quejarse de la vía o del precio. Para no perder eso medimos el sentimiento **por oración** y marcamos las reseñas con alguna oración negativa (`tiene_critica`).
- Las reseñas no traen estrellas, así que el sentimiento sale solo del texto. La validación automática (parte 4) mide **concordancia** con otros métodos, no exactitud. La exactitud real solo se puede dar con la muestra etiquetada a mano (parte 4b).
- Los temas de problema (parte 6b) se detectan con diccionarios de palabras sobre las oraciones negativas: son una aproximación transparente, no un clasificador entrenado.
- Maps solo da fechas relativas (`hace 2 años`); la fecha aproximada sirve para ver el período general, no para análisis mes a mes.
- Maps corta los comentarios largos; esas reseñas quedan marcadas en `truncada`.
- Elegimos pocos lugares por municipio: la muestra no es aleatoria ni representa a todos los visitantes, y Guatavita pesa cerca del 38 % del corpus.
- Las reseñas de Google tienen restricciones de uso: las usamos solo con fines académicos y citando la fuente.

## 0. Configuración

In [ ]:
%%capture
!pip install -q spacy gensim unidecode wordcloud langdetect pysentimiento
!python -m spacy download es_core_news_md

In [ ]:
import os, re, io, sys, json, glob, random, unicodedata, warnings, zipfile, difflib, platform, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import requests
from unidecode import unidecode
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

EN_COLAB = "google.colab" in sys.modules

# Todos los parámetros están aquí para no andar buscándolos por el cuaderno
CONFIG = dict(
    DATA_DIR="datos",
    OUT_DIR="salidas",
    SEED=42,
    # RNT: estados que contamos como prestador vigente
    RNT_ESTADOS_VALIDOS=["ACTIVO"],
    # Reseñas
    MIN_CARACTERES=15, MIN_PALABRAS=3,
    FILTRAR_IDIOMA=True,
    MIN_RESENAS_POR_MUNICIPIO=200,   # solo avisa, no borra nada
    # Texto
    SPACY_MODELO="es_core_news_md",  # en el documento maestro dice _sm; usamos _md porque etiqueta mejor
    REMOVER_TOPONIMOS=True,          # quita nombres de municipios del vocabulario de tópicos
    USAR_BIGRAMAS=True,
    QUITAR_ELOGIOS=True,             # quita adjetivos de elogio genéricos (hermoso, bonito...) del vocabulario de tópicos; el sentimiento usa el texto original
    # Sentimiento
    SENT_BATCH=32, CHUNK_CHARS=350, UMBRAL_SENT=0.25, REUSAR_CACHE=True,
    N_VALIDACION_AUTO=600,           # reseñas que se comparan con el segundo modelo (para que no tarde mucho)
    N_VALIDACION_MANUAL=120,         # reseñas que se sacan para etiquetar a mano (repartidas por estratos)
    MIN_CHARS_ORACION=20,            # las oraciones más cortas se pegan a la vecina para que el modelo tenga contexto
    UMBRAL_P_NEG=0.6,                # una oración cuenta como negativa si P(neg) >= este valor y supera a P(pos)
    ALERTA_MIN_APARICIONES=8, ALERTA_MIN_APARICIONES_MUN=4, ALERTA_MIN_NEG=15,   # mínimos para que una alerta no sea casualidad
    FECHA_EXTRACCION="2026-11-20",   # día en que se sacaron las reseñas; sirve para convertir "hace 2 años" en fecha aproximada
    # LDA
    LDA_K_MIN=3, LDA_K_MAX=12, LDA_K_FIJO=None,   # si no nos convence el k, ponemos un número en LDA_K_FIJO
    LDA_PASSES=10, LDA_MIN_TOKENS=2,
)
SEED = CONFIG["SEED"]; random.seed(SEED); np.random.seed(SEED)

# municipio (sin tildes y en mayúsculas) -> (provincia, nombre para mostrar)
MUNICIPIOS = {
    "ZIPAQUIRA": ("Sabana Centro", "Zipaquirá"),
    "NEMOCON": ("Sabana Centro", "Nemocón"),
    "GUATAVITA": ("Sabana Centro", "Guatavita"),
    "LA MESA": ("Tequendama", "La Mesa"),
    "SAN ANTONIO DEL TEQUENDAMA": ("Tequendama", "San Antonio del Tequendama"),
    "CACHIPAY": ("Tequendama", "Cachipay"),
}
# nombres alternativos con los que aparecen en las fuentes
ALIAS_MUNICIPIO = {"SAN ANTONIO DE TEQUENDAMA": "SAN ANTONIO DEL TEQUENDAMA",
                   "SAN ANTONIO TEQUENDAMA": "SAN ANTONIO DEL TEQUENDAMA",
                   "LAMESA": "LA MESA", "MESA": "LA MESA", "MESA DE YEGUAS": "LA MESA"}
PROVINCIA = {k: v[0] for k, v in MUNICIPIOS.items()}
NOMBRE = {k: v[1] for k, v in MUNICIPIOS.items()}
# nombres para mostrar de los lugares (clave: como sale del nombre del archivo, en minúsculas). Lo que no esté aquí se muestra con mayúscula inicial.
# Revisen y corrijan estos nombres antes de entregar.
LUGARES = {"salto monjas": "Salto de las Monjas", "salto tequendama": "Salto del Tequendama", "mina sal": "Mina de sal",
           "plaza comuneros": "Plaza de los Comuneros", "centro historico": "Centro histórico", "alto mohan": "Alto del Mohán",
           "zoologico santacruz": "Zoológico Santacruz", "catedral": "Catedral de Sal"}

def norm(s):
    # mayúsculas, sin tildes y sin espacios repetidos
    return re.sub(r"\s+", " ", unidecode(str(s)).upper().strip())

def quitar_tildes_keep_n(s):
    # quita tildes pero deja la ñ como está
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn" or c == "̃")
    return unicodedata.normalize("NFC", s)

DATA_DIR = Path(CONFIG["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
OUT = Path(CONFIG["OUT_DIR"]); OUT.mkdir(exist_ok=True)
print("Python", platform.python_version(), "| Colab:", EN_COLAB)

## 1. Registro Nacional de Turismo (RNT)
Lo bajamos de `datos.gov.co`. Primero lo revisamos y después lo limpiamos, porque el RNT trae varios cortes (`ano`, `periodo`) y un mismo prestador puede aparecer repetido.
Si la descarga falla y no hay copia local, el cuaderno sigue sin el RNT (solo se pierde el cruce de la parte 7).

In [ ]:
RNT_URL = "https://www.datos.gov.co/resource/thwd-ivmp.csv"
RNT_RAW = DATA_DIR / "rnt_cundinamarca_raw.csv"

def descargar_rnt(pagina=50000):
    # la API devuelve por páginas; vamos pidiendo hasta que llegue una incompleta
    frames, offset = [], 0
    while True:
        params = {"$where": "upper(departamento)='CUNDINAMARCA'", "$limit": pagina,
                  "$offset": offset, "$order": ":id"}
        r = requests.get(RNT_URL, params=params, timeout=180); r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), dtype=str)   # dtype=str para no dañar códigos ni NIT
        if df.empty: break
        frames.append(df); offset += len(df)
        if len(df) < pagina: break
    return pd.concat(frames, ignore_index=True)

rnt_raw = None
try:
    rnt_raw = descargar_rnt()
    rnt_raw.to_csv(RNT_RAW, index=False)
    print(f"Descargado de datos.gov.co: {len(rnt_raw):,} filas de Cundinamarca")
except Exception as e:
    print("No se pudo descargar el RNT:", e)
    if RNT_RAW.exists():
        rnt_raw = pd.read_csv(RNT_RAW, dtype=str); print("Usamos la copia local:", RNT_RAW)
    else:
        print("Seguimos sin RNT. Si lo quieren, bajen el CSV y guárdenlo como", RNT_RAW)
HAY_RNT = rnt_raw is not None
if HAY_RNT:
    rnt_raw.columns = [c.strip().lower() for c in rnt_raw.columns]
    display(rnt_raw.head(3))

In [ ]:
resumen_rnt = None
if HAY_RNT:
    rnt = rnt_raw.copy()
    rnt["municipio_norm"] = rnt["municipio"].map(norm).replace(ALIAS_MUNICIPIO)
    # avisamos si hay nombres parecidos a los nuestros que se quedaron por fuera
    otros = rnt.loc[~rnt["municipio_norm"].isin(MUNICIPIOS), "municipio_norm"].value_counts()
    parecidos = [m for m in otros.index if any(k.split()[0][:5] in m for k in MUNICIPIOS)]
    if parecidos: print("Nombres parecidos que NO entraron (revisar):", parecidos)
    rnt = rnt[rnt["municipio_norm"].isin(MUNICIPIOS)].copy()

    # revisión antes de limpiar
    print(f"Filas de los 6 municipios: {len(rnt):,}")
    print("Estados:\n", rnt["estado_rnt"].value_counts(dropna=False).to_string())
    rnt["ano_num"] = pd.to_numeric(rnt.get("ano"), errors="coerce")
    if "periodo" in rnt: print("\nCortes (año/periodo):\n", rnt.groupby(["ano", "periodo"]).size().to_string())
    dup = rnt.duplicated(["codigo_rnt", "municipio_norm"], keep=False)
    print(f"\nFilas con código RNT repetido en el mismo municipio: {dup.sum():,}")

    # limpieza: nos quedamos con el corte más reciente de cada prestador y con los estados válidos
    orden = [c for c in ["ano_num", "periodo"] if c in rnt.columns]
    rnt = rnt.sort_values(orden, ascending=False).drop_duplicates(["codigo_rnt", "municipio_norm"], keep="first")
    antes = len(rnt)
    rnt["estado_rnt"] = rnt["estado_rnt"].astype(str).str.upper().str.strip()
    rnt = rnt[rnt["estado_rnt"].isin([e.upper() for e in CONFIG["RNT_ESTADOS_VALIDOS"]])].copy()
    rnt["provincia"] = rnt["municipio_norm"].map(PROVINCIA)
    rnt["municipio_nombre"] = rnt["municipio_norm"].map(NOMBRE)
    print(f"Sin duplicados: {antes:,} | solo estado {CONFIG['RNT_ESTADOS_VALIDOS']}: {len(rnt):,}")
    resumen_rnt = rnt.groupby(["provincia", "municipio_nombre"]).size().rename("prestadores_rnt").reset_index()
    display(resumen_rnt)
    rnt.to_csv(DATA_DIR / "rnt_limpio.csv", index=False)
    for col in ["categoria", "subcategoria", "tipo_rnt"]:      # primera columna de tipo que exista; sirve para el dashboard
        if col in rnt.columns:
            (rnt.groupby(["provincia", "municipio_nombre", col]).size().rename("prestadores").reset_index().rename(columns={col: "categoria"})
                .to_csv(OUT / "rnt_por_categoria.csv", index=False)); print("Prestadores por", col, "guardados en rnt_por_categoria.csv"); break
else:
    print("Parte omitida: no hay RNT.")

## 2. Reseñas: armar el corpus, cargar y limpiar
Subimos el `.zip` con los CSV del scraper (o directamente `resenas_google_maps.csv` si ya está armado). El cuaderno hace solo lo siguiente:
- toma el municipio y el lugar del nombre de cada archivo (`zipaquira_catedral.csv` -> Zipaquirá, catedral),
- se queda con el texto de la reseña y descarta la respuesta del propietario, las filas sin comentario y los datos personales del autor,
- convierte "hace 2 años" en una fecha aproximada, y marca las reseñas que Maps cortó (`truncada`),
- guarda todo en `datos/resenas_google_maps.csv`.
Si más adelante agregamos otra fuente, basta dejar otro `resenas_<fuente>.csv` en `datos/`: se une solo.

In [ ]:
# Solo en Colab: subimos el .zip (o el CSV ya armado). Si ya hay datos en datos/, esta celda no pide nada.
def hay_datos():
    return bool(glob.glob(str(DATA_DIR / "resenas_*.csv")) or glob.glob(str(DATA_DIR / "*.zip")) or glob.glob(str(DATA_DIR / "crudos" / "*.csv")))

if EN_COLAB and not hay_datos():
    from google.colab import files
    subidos = files.upload()
    for nombre, contenido in subidos.items():
        (DATA_DIR / nombre).write_bytes(contenido)
    print("Guardados en", DATA_DIR, ":", list(subidos))
elif hay_datos():
    print("Ya hay datos en", DATA_DIR, "- seguimos.")
else:
    print("Fuera de Colab: copien el .zip o el CSV a", DATA_DIR.resolve())

In [ ]:
# Convierte los CSV crudos de Instant Data Scraper en un solo archivo limpio.
# Los nombres de columna de Maps son códigos raros: "d4r55" es el autor, "wiI7pd" el texto de la reseña
# y "rsqaWe" la fecha relativa. "wiI7pd 2" es la respuesta del propietario y NO se usa.
MUN_ARCHIVO = {"cachipay": "Cachipay", "guatavita": "Guatavita", "lamesa": "La Mesa", "nemocon": "Nemocón",
               "sanantonio": "San Antonio del Tequendama", "zipaquira": "Zipaquirá"}
NUMEROS = {"un": 1, "una": 1}

def fecha_aproximada(rel, ref):
    s = str(rel).lower().replace("editado", "").strip()
    m = re.search(r"(\d+|un|una)\s+(año|años|mes|meses|semana|semanas|día|días|dia|dias)", s)
    if not m: return pd.NaT
    n = NUMEROS.get(m.group(1)) or int(m.group(1)); u = m.group(2)
    dias = 365*n if u.startswith("año") else 30.4*n if u.startswith("mes") else 7*n if u.startswith("semana") else n
    return (ref - pd.Timedelta(days=round(dias))).normalize()

def consolidar_crudos(carpeta, ref):
    partes, resumen = [], []
    for f in sorted(glob.glob(str(carpeta / "*.csv"))):
        clave = Path(f).name.split("_")[0].lower()
        if clave not in MUN_ARCHIVO: print("  Omitido (no reconocemos el municipio):", Path(f).name); continue
        for enc in ("utf-8", "utf-8-sig", "latin-1"):
            try: d = pd.read_csv(f, encoding=enc); break
            except Exception: d = None
        if d is None: print("  Omitido (no se pudo leer):", Path(f).name); continue
        if "d4r55" not in d.columns or "wiI7pd" not in d.columns:
            print("  Omitido (no tiene el formato del scraper):", Path(f).name); continue
        n0 = len(d); d = d[d["d4r55"].notna()]                      # con autor = reseña real
        texto = d["wiI7pd"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
        trunc = texto.str.endswith("…")
        out = pd.DataFrame({"municipio": MUN_ARCHIVO[clave], "atractivo_turistico": Path(f).stem[len(clave)+1:].replace("_", " "),
                            "estrellas": np.nan, "fecha_relativa": d["rsqaWe"].values,
                            "texto_resena": texto.str.rstrip("…").str.strip().values, "truncada": trunc.values,
                            "_autor": d["d4r55"].astype(str).values})
        out["fecha"] = out["fecha_relativa"].map(lambda r: fecha_aproximada(r, ref))
        out = out[out["texto_resena"] != ""]
        out = out.drop_duplicates(["municipio", "atractivo_turistico", "_autor", "texto_resena"])
        resumen.append((Path(f).name, n0, len(out))); partes.append(out)
    df = pd.concat(partes, ignore_index=True)
    df = df.drop_duplicates(["municipio", "_autor", "texto_resena"])
    df["fuente"] = "google_maps"
    df = df.drop(columns="_autor")[["municipio","atractivo_turistico","estrellas","fecha","texto_resena","fuente","truncada","fecha_relativa"]]
    print(pd.DataFrame(resumen, columns=["archivo", "filas", "reseñas con texto"]).to_string(index=False))
    return df

destino = DATA_DIR / "resenas_google_maps.csv"
CRUDOS = DATA_DIR / "crudos"
for z in glob.glob(str(DATA_DIR / "*.zip")):                       # si subimos un zip, lo abrimos
    CRUDOS.mkdir(exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        for nombre in zf.namelist():
            if nombre.lower().endswith(".csv") and not nombre.startswith("__MACOSX"):
                (CRUDOS / Path(nombre).name).write_bytes(zf.read(nombre))
if not destino.exists() and CRUDOS.exists() and glob.glob(str(CRUDOS / "*.csv")):
    corpus_gm = consolidar_crudos(CRUDOS, pd.Timestamp(CONFIG["FECHA_EXTRACCION"]))
    corpus_gm.to_csv(destino, index=False, encoding="utf-8")
    print(f"\nCorpus armado: {len(corpus_gm):,} reseñas -> {destino}")
else:
    print("Usamos el archivo que ya existe:", destino if destino.exists() else "(ninguno todavía)")

In [ ]:
# Nombres de columna que aceptamos para cada campo (por si otra fuente los trae distintos)
ALIAS_COLS = {
    "texto_resena": ["texto_resena","texto","resena","reseña","review","comentario","comment","text","opinion","contenido","review_text"],
    "estrellas": ["estrellas","rating","calificacion","calificación","puntuacion","puntuación","stars","score","nota"],
    "municipio": ["municipio","ciudad","localidad","city","town","ubicacion"],
    "fecha": ["fecha","date","fecha_resena","fecha_publicacion","published","published_date"],
    "atractivo_turistico": ["atractivo_turistico","atractivo","lugar","sitio","place","nombre_lugar","establecimiento","place_name"],
    "fuente": ["fuente","source","plataforma"],
}
def leer_tabla(path):
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xls"): return pd.read_excel(p)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try: return pd.read_csv(p, encoding=enc, sep=None, engine="python")
        except Exception: continue
    raise ValueError(f"No pudimos leer {p}")

def estandarizar_columnas(df, nombre_archivo):
    df = df.rename(columns={c: c.strip().lower() for c in df.columns})
    ren = {}
    for canon, alias in ALIAS_COLS.items():
        for a in alias:
            if a in df.columns and canon not in ren.values():
                ren[a] = canon; break
    df = df.rename(columns=ren)
    if "fuente" not in df.columns: df["fuente"] = Path(nombre_archivo).stem.replace("resenas_", "")
    faltan = [c for c in ["texto_resena", "municipio"] if c not in df.columns]
    if faltan: raise ValueError(f"{nombre_archivo}: faltan columnas {faltan}. Tiene: {list(df.columns)}")
    for c in ["estrellas", "fecha", "atractivo_turistico"]:
        if c not in df.columns: df[c] = np.nan
    if "truncada" not in df.columns: df["truncada"] = False
    return df[["municipio","atractivo_turistico","estrellas","fecha","texto_resena","fuente","truncada"]].copy()

archivos = sorted(glob.glob(str(DATA_DIR / "resenas_*.csv")) + glob.glob(str(DATA_DIR / "resenas_*.xlsx")))
if not archivos:
    raise FileNotFoundError("No hay reseñas en datos/ (el archivo debe llamarse resenas_*.csv). "
                            "Suban resenas_google_maps.csv con la celda de arriba.")
partes = [estandarizar_columnas(leer_tabla(a), a) for a in archivos]
for a, p in zip(archivos, partes): print(f"  {Path(a).name}: {len(p):,} filas")
resenas_raw = pd.concat(partes, ignore_index=True)
print(f"\nReseñas crudas: {len(resenas_raw):,}")

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0          # para que la detección de idioma dé siempre lo mismo
embudo = [("crudas", len(resenas_raw))]
df = resenas_raw.copy()

# 1) Texto: quitamos HTML, las marcas "(Traducido por Google)" y espacios de más
def limpiar_texto_crudo(t):
    if pd.isna(t): return ""
    t = str(t)
    m = re.search(r"\((?:traducido por google|translated by google)\)\s*(.*?)\s*\(original\)", t, flags=re.I | re.S)
    if m: t = m.group(1)
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"\s+", " ", t).strip()
df["texto_resena"] = df["texto_resena"].map(limpiar_texto_crudo)

# 2) Municipio -> nombre canónico (alias, subcadena y coincidencia aproximada)
def mapear_municipio(x):
    n = norm(x); n = ALIAS_MUNICIPIO.get(n, n)
    if n in MUNICIPIOS: return n
    for k in sorted(MUNICIPIOS, key=len, reverse=True):
        if k in n: return k
    m = difflib.get_close_matches(n, list(MUNICIPIOS), n=1, cutoff=0.85)
    return m[0] if m else None
df["municipio_norm"] = df["municipio"].map(mapear_municipio)
no_map = df.loc[df["municipio_norm"].isna(), "municipio"].value_counts().head(10)
if len(no_map): print("Municipios sin reconocer (se descartan):\n", no_map.to_string())
df = df.dropna(subset=["municipio_norm"]); embudo.append(("municipio válido", len(df)))

# 3) Estrellas (puede venir vacía, como en nuestro caso) y fecha
def a_estrellas(x):
    if pd.isna(x): return np.nan
    m = re.search(r"\d+(?:[.,]\d+)?", str(x))
    v = float(m.group().replace(",", ".")) if m else np.nan
    return v if 1 <= v <= 5 else np.nan
df["estrellas"] = df["estrellas"].map(a_estrellas)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", utc=True).dt.tz_localize(None)
df.loc[df["fecha"] > pd.Timestamp.today(), "fecha"] = pd.NaT

# 4) Longitud mínima
n_pal = df["texto_resena"].str.split().str.len()
df = df[(df["texto_resena"].str.len() >= CONFIG["MIN_CARACTERES"]) & (n_pal >= CONFIG["MIN_PALABRAS"])]
embudo.append(("longitud mínima", len(df)))

# 5) Duplicados: mismo municipio y mismo texto (sin tildes ni signos)
df["_clave"] = df["texto_resena"].map(lambda t: re.sub(r"[^a-z0-9 ]", "", unidecode(t.lower())))
df = df.drop_duplicates(["municipio_norm", "_clave"]).drop(columns="_clave"); embudo.append(("sin duplicados", len(df)))

# 6) Idioma: nos quedamos con español
def es_espanol(t):
    if len(t) < 25: return True       # textos muy cortos no se pueden detectar bien, los dejamos
    try: return detect(t) == "es"
    except Exception: return True
if CONFIG["FILTRAR_IDIOMA"]:
    ok = df["texto_resena"].map(es_espanol)
    print(f"Reseñas que no están en español: {(~ok).sum():,}")
    df = df[ok]; embudo.append(("en español", len(df)))

df["provincia"] = df["municipio_norm"].map(PROVINCIA); df["municipio_nombre"] = df["municipio_norm"].map(NOMBRE)
df = df.reset_index(drop=True); df.insert(0, "id_resena", np.arange(1, len(df) + 1))
resenas = df
resenas["lugar_nombre"] = resenas["atractivo_turistico"].map(lambda x: LUGARES.get(str(x).strip().lower(), str(x).strip().title()))
display(pd.DataFrame(embudo, columns=["paso", "filas"]).assign(perdidas=lambda d: d["filas"].diff().fillna(0).astype(int)))
resenas.to_csv(OUT / "resenas_limpias.csv", index=False)

In [ ]:
# Revisión rápida de cómo quedó el corpus
print("Reseñas por municipio y fuente:")
display(pd.crosstab(resenas["municipio_nombre"], resenas["fuente"], margins=True))
cnt = resenas["municipio_nombre"].value_counts()
for m, n in cnt.items():
    if n < CONFIG["MIN_RESENAS_POR_MUNICIPIO"]:
        print(f"  Ojo: {m} tiene solo {n} reseñas (menos de {CONFIG['MIN_RESENAS_POR_MUNICIPIO']}), sus conclusiones son frágiles.")
faltan = [v[1] for v in MUNICIPIOS.values() if v[1] not in cnt.index]
if faltan: print("  Municipios sin reseñas:", faltan)
print("\nFechas vacías: %.1f%% | reseñas truncadas por Maps: %.1f%%" % (resenas["fecha"].isna().mean()*100, resenas["truncada"].astype(bool).mean()*100))
if resenas["fecha"].notna().any(): print("Rango de fechas (aproximado):", resenas["fecha"].min().date(), "->", resenas["fecha"].max().date())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cnt.sort_values().plot(kind="barh", ax=ax[0], color="#3b7dd8"); ax[0].set_title("Reseñas por municipio"); ax[0].set_ylabel("")
resenas["atractivo_turistico"].value_counts().head(12).sort_values().plot(kind="barh", ax=ax[1], color="#7a9e7e")
ax[1].set_title("Lugares con más reseñas"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

## 3. Preparar el texto (NLP)
Pasos que seguimos con cada reseña:
- Dejamos el texto con tildes para que spaCy etiquete mejor y solo normalizamos el vocabulario al final.
- Nos quedamos con sustantivos, adjetivos y verbos.
- Si spaCy devuelve un lema con varias palabras (por ejemplo `desconectar él`), tomamos solo la primera.
- Corregimos a mano algunos lemas que spaCy hace mal (`vistas` salía como `visto`).
- Quitamos stopwords, palabras demasiado generales (`lugar`, `sitio`...) y los nombres de los municipios, para que no dominen los tópicos.
- Buscamos bigramas como `catedral_sal` con gensim.

In [ ]:
import spacy
def cargar_spacy(nombre):
    try: return spacy.load(nombre, disable=["parser", "ner"])
    except OSError:
        import subprocess; subprocess.run([sys.executable, "-m", "spacy", "download", nombre], check=True)
        return spacy.load(nombre, disable=["parser", "ner"])
nlp = cargar_spacy(CONFIG["SPACY_MODELO"])

POS_VALIDOS = {"NOUN", "ADJ", "VERB"}
# palabras que aparecen en todas partes y no dicen nada del municipio
STOP_DOMINIO = {"lugar","sitio","vez","cosa","ir","ser","estar","haber","tener","hacer","poder","dar","ver","quedar",
                "visitar","recomendar","conocer","llegar","salir","decir","querer","parecer","pasar","encontrar","ano","dia",
                "gracias","gustar","duda","entender","contar","tomar","dejar","momento","especial","persona","gente","grupo"}
TOPONIMOS = {"zipaquira","nemocon","guatavita","mesa","cachipay","tequendama","antonio","cundinamarca","sabana","bogota","tomine","sesquile",
             "zipaquier","cundinamarco","guatavitir","nemoconer"}   # incluye lemas raros que spaCy saca de gentilicios
# elogios genéricos: aparecen en casi todas las reseñas y tapan los temas de verdad
ELOGIOS = {"hermoso","bonito","lindo","bello","excelente","increible","espectacular","precioso","maravilloso","agradable",
           "recomendado","experiencia","encantar","unico","perfecto","genial","chevere","magnifico","fantastico","divino"}
LEMMA_FIX = {"vista":"vista","vistas":"vista","guias":"guia","vias":"via","paisajes":"paisaje","cascadas":"cascada","miradores":"mirador"}

STOP_FINAL = {quitar_tildes_keep_n(w.lower()) for w in nlp.Defaults.stop_words} | STOP_DOMINIO
if CONFIG["REMOVER_TOPONIMOS"]: STOP_FINAL |= TOPONIMOS
if CONFIG["QUITAR_ELOGIOS"]: STOP_FINAL |= ELOGIOS

def texto_para_spacy(t):
    t = re.sub(r"http\S+|www\.\S+|\S+@\S+", " ", t)
    return re.sub(r"\s+", " ", t.lower()).strip()

def tokenizar(textos, batch=256):
    salida = []
    for doc in nlp.pipe((texto_para_spacy(t) for t in textos), batch_size=batch):
        toks = []
        for t in doc:
            if t.pos_ not in POS_VALIDOS or t.is_stop or not t.is_alpha: continue
            lem = LEMMA_FIX.get(quitar_tildes_keep_n(t.text), t.lemma_.split()[0])
            lem = quitar_tildes_keep_n(lem.lower())
            if len(lem) < 3 or lem in STOP_FINAL: continue
            toks.append(lem)
        salida.append(toks)
    return salida

resenas["tokens"] = tokenizar(resenas["texto_resena"].tolist())

frases = None
if CONFIG["USAR_BIGRAMAS"]:
    from gensim.models.phrases import Phrases
    frases = Phrases(resenas["tokens"], min_count=8, threshold=0.55, scoring="npmi").freeze()
    resenas["tokens"] = [frases[t] for t in resenas["tokens"]]
    print("Bigramas que encontró (muestra):", sorted({w for t in resenas["tokens"] for w in t if "_" in w})[:20])

def con_bigramas(listas):
    # aplica los mismos bigramas a otros textos (por ejemplo, las oraciones de la parte 5)
    return [frases[t] for t in listas] if frases is not None else listas

resenas["texto_nlp"] = resenas["tokens"].map(" ".join)
resenas["n_tokens"] = resenas["tokens"].map(len)
print(f"Reseñas sin ningún token útil: {(resenas['n_tokens']==0).sum():,} | mediana de tokens: {resenas['n_tokens'].median():.0f}")
display(resenas[["municipio_nombre", "texto_resena", "texto_nlp"]].sample(6, random_state=SEED))

## 4. Sentimiento (de −1 a +1), oración por oración
Usamos **`pysentimiento`**, un modelo ya entrenado para español. Cada reseña se parte en **oraciones y cláusulas** (se corta también antes de `pero`, `aunque`, `lo malo`...), el modelo puntúa cada trozo con `P(positivo) − P(negativo)` y la reseña queda con el promedio ponderado por longitud.

¿Por qué por oración? Con la reseña completa casi todo cae en +0,9: *"Excelente lugar, pero la carretera está pésima"* se lee como positiva y la queja desaparece. Por oración conservamos tres cosas nuevas:
- `sent_min`: la oración más negativa de la reseña,
- `tiene_critica`: la reseña tiene al menos una oración claramente negativa,
- `mixta`: tiene oraciones positivas y negativas a la vez.

**Validación automática:** como las reseñas no traen estrellas, medimos la **concordancia** del modelo con dos métodos independientes: (1) reglas simples de palabras y emojis, y (2) otro modelo (`nlptown/bert-base-multilingual-uncased-sentiment`, de 1 a 5 estrellas). Que coincidan nos da confianza, pero no equivale a exactitud contra etiquetas humanas; por eso en la parte 4b se genera una muestra para etiquetar a mano.
Si el modelo principal no se puede cargar, el cuaderno se detiene: preferimos no inventar un puntaje.

In [ ]:
from tqdm.auto import tqdm

def trocear(texto, max_chars):
    # parte el texto en trozos de máximo max_chars, cortando en fin de oración cuando se puede
    oraciones = re.split(r"(?<=[\.\!\?\n])\s+", texto); chunks, actual = [], ""
    for o in oraciones:
        if len(actual) + len(o) + 1 <= max_chars: actual = (actual + " " + o).strip(); continue
        if actual: chunks.append(actual)
        while len(o) > max_chars:
            corte = o.rfind(" ", 0, max_chars); corte = corte if corte > 0 else max_chars
            chunks.append(o[:corte]); o = o[corte:].strip()
        actual = o
    if actual: chunks.append(actual)
    return chunks or [texto]

# Marcadores de contraste: cortamos justo antes porque suelen separar lo bueno de lo malo
MARCADORES = r"(?=\b(?:pero|aunque|sin embargo|no obstante|lo malo|lo negativo|lastima|lástima|eso si|eso sí|solo que|excepto|desafortunadamente)\b)"

def dividir_oraciones(texto):
    t = re.sub(r"http\S+|www\.\S+", " ", texto); t = re.sub(r"\s+", " ", t).strip()
    partes = []
    for o in re.split(r"(?<=[\.\!\?;])\s+|\n", t):
        partes += [p.strip(" ,") for p in re.split(MARCADORES, o, flags=re.I)]
    partes = [p for p in partes if p]
    # los fragmentos muy cortos se pegan al vecino para que el modelo tenga contexto
    out = []
    for p in partes:
        if out and len(out[-1]) < CONFIG["MIN_CHARS_ORACION"]: out[-1] = out[-1] + " " + p
        else: out.append(p)
    if len(out) > 1 and len(out[-1]) < CONFIG["MIN_CHARS_ORACION"]:
        ultima = out.pop(); out[-1] += " " + ultima
    final = []
    for p in out: final += trocear(p, CONFIG["CHUNK_CHARS"])      # tope de seguridad por longitud
    return final or [t]

def sentimiento_oraciones(df):
    from pysentimiento import create_analyzer
    analyzer = create_analyzer(task="sentiment", lang="es")
    filas = [(rid, j, o) for rid, t in zip(df["id_resena"], df["texto_resena"]) for j, o in enumerate(dividir_oraciones(t))]
    ors = pd.DataFrame(filas, columns=["id_resena", "n_oracion", "oracion"])
    probs = []
    for ini in tqdm(range(0, len(ors), CONFIG["SENT_BATCH"]), desc="Sentimiento por oración"):
        for r in analyzer.predict(ors["oracion"].iloc[ini: ini + CONFIG["SENT_BATCH"]].tolist()):
            probs.append([r.probas.get("POS", 0), r.probas.get("NEG", 0), r.probas.get("NEU", 0)])
    ors[["p_pos", "p_neg", "p_neu"]] = np.array(probs)
    return ors

# La caché guarda el resultado (el cálculo es lento); solo se reutiliza si los textos y las reglas de corte son idénticos
VERSION_DIVISOR = f"v2|{MARCADORES}|{CONFIG['MIN_CHARS_ORACION']}|{CONFIG['CHUNK_CHARS']}|"
huellas = resenas.set_index("id_resena")["texto_resena"].map(lambda t: hashlib.md5((VERSION_DIVISOR + t).encode("utf-8")).hexdigest()[:12])
cache = OUT / "cache_sentimiento_oraciones.csv"
oraciones = None
if CONFIG["REUSAR_CACHE"] and cache.exists():
    c = pd.read_csv(cache, dtype={"huella": str})
    if {"huella", "id_resena", "oracion", "p_pos", "p_neg", "p_neu"} <= set(c.columns):
        h_c = c.drop_duplicates("id_resena").set_index("id_resena")["huella"].sort_index()
        if h_c.index.equals(huellas.sort_index().index) and (h_c.values == huellas.sort_index().values).all():
            oraciones = c.drop(columns="huella"); print("Sentimiento recuperado de la caché")
if oraciones is None:
    oraciones = sentimiento_oraciones(resenas)
    oraciones.assign(huella=oraciones["id_resena"].map(huellas)).to_csv(cache, index=False)

# Por oración: puntaje y marca de oración negativa/positiva
oraciones["oracion"] = oraciones["oracion"].fillna("").astype(str)
oraciones["score"] = oraciones["p_pos"] - oraciones["p_neg"]
oraciones["peso"] = oraciones["oracion"].str.len().clip(lower=1)
oraciones["es_negativa"] = (oraciones["p_neg"] >= CONFIG["UMBRAL_P_NEG"]) & (oraciones["p_neg"] > oraciones["p_pos"])
oraciones["es_positiva"] = (oraciones["p_pos"] >= CONFIG["UMBRAL_P_NEG"]) & (oraciones["p_pos"] > oraciones["p_neg"])

# Por reseña: promedio ponderado por longitud + marcas de crítica
for c_ in ["p_pos", "p_neg", "p_neu"]: oraciones[c_ + "_w"] = oraciones[c_] * oraciones["peso"]
g = oraciones.groupby("id_resena")
sent = g[["p_pos_w", "p_neg_w", "p_neu_w", "peso"]].sum()
for c_ in ["p_pos", "p_neg", "p_neu"]: sent[c_] = sent[c_ + "_w"] / sent["peso"]
sent = sent[["p_pos", "p_neg", "p_neu"]]
sent["sent_min"] = g["score"].min(); sent["n_oraciones"] = g.size()
sent["n_oraciones_neg"] = g["es_negativa"].sum(); sent["n_oraciones_pos"] = g["es_positiva"].sum()
oraciones = oraciones.drop(columns=["p_pos_w", "p_neg_w", "p_neu_w"])

resenas = resenas.drop(columns=[c_ for c_ in sent.columns if c_ in resenas.columns]).merge(sent, left_on="id_resena", right_index=True, how="left").reset_index(drop=True)
resenas["sent_metodo"] = "pysentimiento (por oración)"
resenas["sentimiento"] = resenas["p_pos"] - resenas["p_neg"]
u = CONFIG["UMBRAL_SENT"]
resenas["sent_label"] = np.select([resenas["sentimiento"] >= u, resenas["sentimiento"] <= -u], ["positivo", "negativo"], "neutro")
resenas["tiene_critica"] = resenas["n_oraciones_neg"] > 0
resenas["mixta"] = resenas["tiene_critica"] & (resenas["n_oraciones_pos"] > 0)
oraciones = oraciones.merge(resenas[["id_resena", "provincia", "municipio_nombre", "atractivo_turistico", "lugar_nombre"]], on="id_resena", how="left")

print(f"Oraciones analizadas: {len(oraciones):,} (promedio {len(oraciones)/len(resenas):.1f} por reseña)")
display(pd.DataFrame({"proporción": resenas["sent_label"].value_counts(normalize=True).round(3)}))
print("Reseñas con al menos una oración negativa: %.1f%% | mixtas (positivas y negativas a la vez): %.1f%%" % (resenas["tiene_critica"].mean()*100, resenas["mixta"].mean()*100))
print("Reseñas con |sentimiento| > 0,9: %.1f%% (con la reseña completa, antes, era ~75%%)" % ((resenas["sentimiento"].abs() > 0.9).mean()*100))
plt.figure(figsize=(7, 3)); plt.hist(resenas["sentimiento"], bins=40, color="#3b7dd8"); plt.axvline(0, c="k", lw=.8)
plt.title("Distribución del sentimiento por reseña"); plt.xlabel("P(pos) − P(neg)"); plt.tight_layout(); plt.show()

In [ ]:
# Validación automática del sentimiento (concordancia con métodos independientes)
from sklearn.metrics import cohen_kappa_score, classification_report
from scipy.stats import spearmanr

# Método 1: reglas simples con palabras y emojis. Solo opina cuando hay señal clara.
POS_W = ["excelente","hermos","bonit","lind","bell","recomend","encant","increíble","increible","espectacular","maravill","genial",
         "perfecto","amable","limpio","tranquil","agradable","fantástic","fantastic","delicios","vale la pena","gracias","disfrut","feliz"]
NEG_W = ["pésim","pesim","malo","mala","horrible","sucio","caro","costos","decepcion","aburrid","peligros","abandonad","descuidad",
         "demor","grosero","mal estado","no vale","no recomiendo","feo","fea","estafa","basura","desorden","deplorable","terrible"]
EMO_POS = set("😍😊👍😀🥰✨👏💖😁🙌😃😄🤩💕👌🌟😘❤")
EMO_NEG = set("😡👎😞😠😢🤮💔😤😒😖😩")
def lexico(t):
    t = t.lower()
    p = sum(w in t for w in POS_W) + sum(e in t for e in EMO_POS)
    n = sum(w in t for w in NEG_W) + sum(e in t for e in EMO_NEG)
    return None if p == n else ("positivo" if p > n else "negativo")

lex = resenas["texto_resena"].map(lexico)
con_senal = lex.notna()
modelo_pol = np.where(resenas["sentimiento"] > 0, "positivo", "negativo")   # polaridad del modelo, sin zona neutra
print(f"[Reglas] reseñas con señal clara: {con_senal.sum():,} de {len(resenas):,} ({con_senal.mean():.0%})")
if con_senal.sum() >= 30:
    a, b = lex[con_senal].values, modelo_pol[con_senal.values]
    print("[Reglas] acuerdo de polaridad: %.1f%% | kappa de Cohen: %.3f" % ((a == b).mean()*100, cohen_kappa_score(a, b)))
    print(classification_report(a, b, zero_division=0))

# Método 2: otro modelo que da estrellas de 1 a 5 (sobre una muestra para que no tarde demasiado)
muestra = resenas.sample(min(CONFIG["N_VALIDACION_AUTO"], len(resenas)), random_state=SEED)
try:
    from transformers import pipeline
    clf = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", truncation=True, max_length=256)
    salida = clf(muestra["texto_resena"].tolist(), batch_size=16)
    est2 = np.array([int(r["label"].split()[0]) for r in salida])
    lab2 = np.select([est2 <= 2, est2 >= 4], ["negativo", "positivo"], "neutro")
    rho, p = spearmanr(muestra["sentimiento"], est2)
    print(f"\n[Segundo modelo] muestra de {len(muestra)} reseñas")
    print(f"[Segundo modelo] Spearman entre nuestro sentimiento y sus estrellas estimadas: {rho:.3f} (p={p:.2g})")
    print("[Segundo modelo] acuerdo de etiquetas: %.1f%% | kappa: %.3f" % ((lab2 == muestra["sent_label"].values).mean()*100, cohen_kappa_score(lab2, muestra["sent_label"].values)))
    display(pd.crosstab(pd.Series(lab2, name="segundo modelo"), muestra["sent_label"].reset_index(drop=True).rename("nuestro modelo"), normalize="index").round(2))
except Exception as e:
    print(f"\n[Segundo modelo] no se pudo ejecutar ({type(e).__name__}); seguimos solo con la validación por reglas.")

# Las críticas: ¿el modelo las detecta cuando hay palabras negativas claras?
con_neg = resenas["texto_resena"].map(lambda t: any(w in t.lower() for w in NEG_W))
print("\n[Críticas] reseñas con alguna palabra negativa de las reglas: %.1f%%" % (con_neg.mean()*100))
print("[Críticas] de esas, marcadas con alguna oración negativa por el modelo: %.1f%% | entre las que no tienen palabras negativas: %.1f%%"
      % (resenas.loc[con_neg, "tiene_critica"].mean()*100, resenas.loc[~con_neg, "tiene_critica"].mean()*100))

## 4b. Validación manual (la única que mide exactitud de verdad)
Las validaciones automáticas de arriba solo miden concordancia. Para poder decir en el informe *"el modelo acierta X %"* hay que comparar con etiquetas hechas por personas. El cuaderno deja listo el material:
1. Se genera `salidas/muestra_validacion_manual.csv`: unas 120 reseñas, tomadas por estratos (negativas, neutras o mixtas, positivas) y **sin mostrar la etiqueta del modelo** para no sesgar. Las repartimos entre los 4 integrantes.
2. Cada quien llena la columna `etiqueta_humana` con `positivo`, `neutro`, `negativo` o `mixto`, y el archivo se guarda como `datos/validacion_manual.csv`.
3. Al volver a correr esta celda, se calculan acuerdo, kappa y precisión/recall por clase. Como la muestra es estratificada, el acierto global se reajusta con el tamaño real de cada estrato.

Mientras ese archivo no exista, la celda solo prepara la muestra y no inventa ninguna métrica.

In [ ]:
def estrato(r):
    if r["sent_label"] == "negativo": return "negativo"
    if r["sent_label"] == "neutro" or r["mixta"]: return "neutro o mixto"
    return "positivo"
resenas["_estrato"] = resenas.apply(estrato, axis=1)
tam_estrato = resenas["_estrato"].value_counts()
n_g = max(1, CONFIG["N_VALIDACION_MANUAL"] // 3)
mv = pd.concat([g_.sample(min(n_g, len(g_)), random_state=SEED) for _, g_ in resenas.groupby("_estrato")]).sample(frac=1, random_state=SEED)
mv[["id_resena", "municipio_nombre", "texto_resena"]].assign(etiqueta_humana="").to_csv(OUT / "muestra_validacion_manual.csv", index=False, encoding="utf-8-sig")
mv[["id_resena", "sent_label", "sentimiento", "tiene_critica", "_estrato"]].rename(columns={"_estrato": "estrato"}).to_csv(OUT / "muestra_validacion_clave.csv", index=False)
print(f"Muestra para etiquetar a mano: {len(mv)} reseñas -> {OUT/'muestra_validacion_manual.csv'}")

ETIQ = DATA_DIR / "validacion_manual.csv"
if ETIQ.exists():
    h = pd.read_csv(ETIQ, encoding="utf-8-sig")
    h["etiqueta_humana"] = h["etiqueta_humana"].astype(str).str.lower().str.strip()
    h = h[h["etiqueta_humana"].isin(["positivo", "neutro", "negativo", "mixto"])]
    clave = mv[["id_resena", "sent_label", "_estrato"]]
    h = h.merge(clave, on="id_resena", how="inner")
    if len(h) >= 30:
        hum = h["etiqueta_humana"].replace({"mixto": "neutro"})          # para comparar en 3 clases, "mixto" cuenta como neutro
        print(f"\n[Manual] reseñas etiquetadas: {len(h)}")
        print("[Manual] acuerdo con el modelo (muestra): %.1f%% | kappa de Cohen: %.3f" % ((hum == h['sent_label']).mean()*100, cohen_kappa_score(hum, h['sent_label'])))
        pesos = h["_estrato"].map(tam_estrato / h["_estrato"].value_counts())   # reajuste por estrato
        print("[Manual] acuerdo ajustado al corpus completo: %.1f%%" % (np.average(hum == h['sent_label'], weights=pesos)*100))
        print(classification_report(hum, h["sent_label"], zero_division=0))
        display(pd.crosstab(h["etiqueta_humana"], h["sent_label"]))
    else:
        print(f"[Manual] {ETIQ.name} tiene solo {len(h)} filas válidas; hacen falta al menos 30.")
else:
    print(f"[Manual] todavía no hay {ETIQ}. Etiqueten la muestra y súbanla ahí para medir la exactitud real.")
resenas = resenas.drop(columns="_estrato")

In [ ]:
# Polaridad por municipio y provincia, con intervalo de confianza del 95% (bootstrap)
from scipy.stats import mannwhitneyu
rng = np.random.default_rng(SEED)
def ic95(x, B=1000):
    x = np.asarray(x)
    if len(x) < 2: return (np.nan, np.nan)
    m = rng.choice(x, (B, len(x))).mean(axis=1); return tuple(np.percentile(m, [2.5, 97.5]))
def resumen_sent(g):
    lo, hi = ic95(g["sentimiento"])
    trunc = g["truncada"].astype(bool)
    return pd.Series({"n_resenas": len(g), "sent_medio": g["sentimiento"].mean(), "ic95_inf": lo, "ic95_sup": hi,
                      "pct_positivo": (g["sent_label"] == "positivo").mean(), "pct_neutro": (g["sent_label"] == "neutro").mean(),
                      "pct_negativo": (g["sent_label"] == "negativo").mean(),
                      "pct_con_critica": g["tiene_critica"].mean(),          # reseñas con al menos una oración negativa
                      "pct_truncadas": trunc.mean(), "sent_medio_sin_truncadas": g.loc[~trunc, "sentimiento"].mean(),
                      "estrellas_medias": g["estrellas"].mean()})
met_mun = resenas.groupby(["provincia", "municipio_nombre"]).apply(resumen_sent).reset_index()
met_prov = resenas.groupby("provincia").apply(resumen_sent).reset_index()
# Sensibilidad: Guatavita pesa mucho en Sabana Centro; con "equilibrado" cada municipio cuenta igual
met_prov["sent_medio_equilibrado"] = met_prov["provincia"].map(met_mun.groupby("provincia")["sent_medio"].mean())
display(met_mun.round(3)); display(met_prov.round(3))

# Diferencia entre provincias con su intervalo (bootstrap) y prueba de Mann-Whitney
a = resenas.loc[resenas["provincia"] == "Sabana Centro", "sentimiento"].values
b = resenas.loc[resenas["provincia"] == "Tequendama", "sentimiento"].values
dif = a.mean() - b.mean()
boots = rng.choice(a, (1000, len(a))).mean(axis=1) - rng.choice(b, (1000, len(b))).mean(axis=1)
p_mw = mannwhitneyu(a, b, alternative="two-sided").pvalue
dif_prov = pd.DataFrame([dict(comparacion="Sabana Centro − Tequendama", diferencia=dif, ic95_inf=np.percentile(boots, 2.5),
                              ic95_sup=np.percentile(boots, 97.5), p_mann_whitney=p_mw,
                              diferencia_equilibrada=met_prov.set_index("provincia").loc["Sabana Centro", "sent_medio_equilibrado"]
                                                     - met_prov.set_index("provincia").loc["Tequendama", "sent_medio_equilibrado"])])
display(dif_prov.round(4)); dif_prov.to_csv(OUT / "diferencia_provincias.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
orden = met_mun.sort_values("sent_medio")
ax[0].barh(orden["municipio_nombre"], orden["sent_medio"], xerr=[orden["sent_medio"]-orden["ic95_inf"], orden["ic95_sup"]-orden["sent_medio"]], color="#3b7dd8")
ax[0].axvline(0, c="k", lw=.8); ax[0].set_title("Polaridad media (IC 95%)"); ax[0].set_xlim(-1, 1)
resenas.boxplot(column="sentimiento", by="municipio_nombre", ax=ax[1], rot=25); ax[1].set_title("Distribución por municipio"); plt.suptitle("")
plt.tight_layout(); plt.show()

# Por lugar: sirve para ver qué sitios puntuales pesan en cada municipio
por_lugar = (resenas.groupby(["municipio_nombre", "atractivo_turistico", "lugar_nombre"])
             .agg(n=("sentimiento", "size"), sent_medio=("sentimiento", "mean"), pct_negativo=("sent_label", lambda s: (s == "negativo").mean()),
                  pct_con_critica=("tiene_critica", "mean")).round(3).reset_index())
por_lugar["muestra_pequena"] = por_lugar["n"] < 50       # con menos de 50 reseñas el promedio es poco estable
display(por_lugar.sort_values(["municipio_nombre", "sent_medio"]))
por_lugar.to_csv(OUT / "sentimiento_por_lugar.csv", index=False)

# Por año (la fecha es aproximada: sirve para tendencia general, no para meses)
por_anio = (resenas.assign(anio=resenas["fecha"].dt.year).dropna(subset=["anio"]).groupby(["provincia", "anio"])
            .agg(n=("sentimiento", "size"), sent_medio=("sentimiento", "mean"), pct_con_critica=("tiene_critica", "mean")).round(3).reset_index())
por_anio["anio"] = por_anio["anio"].astype(int)
por_anio = por_anio[por_anio["n"] >= 30]
por_anio.to_csv(OUT / "sentimiento_por_anio.csv", index=False)

## 5. TF-IDF: términos distintivos y alertas
- **Por municipio:** tratamos a cada municipio como un solo documento; TF-IDF resalta lo que lo distingue de los demás.
- **Alertas:** en las **oraciones negativas** (no en reseñas completas, para no perder quejas escondidas en reseñas positivas), buscamos palabras que aparecen mucho más que en el resto (*lift*), pidiendo un mínimo de apariciones para que no sea casualidad. Cada alerta trae un ejemplo real para interpretarla.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import joblib

# (a) términos distintivos por municipio
docs_mun = resenas.groupby("municipio_nombre")["texto_nlp"].apply(" ".join)
try:   # max_df=0.67 descarta lo que aparece en más de 4 de los 6 municipios
    tfidf_mun = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True, max_df=0.67, min_df=1)
    M = tfidf_mun.fit_transform(docs_mun)
except ValueError:   # si casi todo el vocabulario es compartido, no podamos nada
    print("  Casi todo el vocabulario es compartido; usamos max_df=1.0")
    tfidf_mun = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True, max_df=1.0, min_df=1)
    M = tfidf_mun.fit_transform(docs_mun)
vocab = np.array(tfidf_mun.get_feature_names_out())
filas = []
for i, mun in enumerate(docs_mun.index):
    fila = M[i].toarray().ravel(); top = fila.argsort()[::-1][:20]
    filas += [dict(municipio=mun, termino=vocab[j], tfidf=fila[j]) for j in top if fila[j] > 0]
tfidf_terminos = pd.DataFrame(filas)
display(tfidf_terminos.groupby("municipio").head(8).groupby("municipio")["termino"].apply(lambda s: ", ".join(s)).to_frame("términos más distintivos"))
tfidf_terminos.to_csv(OUT / "tfidf_terminos_municipio.csv", index=False)

# (b) matriz TF-IDF por reseña, por si después queremos un modelo supervisado
tfidf_resena = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=5, max_df=0.8, sublinear_tf=True)
X_tfidf = tfidf_resena.fit_transform(resenas["texto_nlp"]); print("Matriz TF-IDF por reseña:", X_tfidf.shape)
joblib.dump(tfidf_resena, OUT / "tfidf_vectorizer.joblib")

# (c) alertas: lift de cada término en las ORACIONES negativas (no en reseñas completas), por provincia y por municipio.
# Así también cuentan las quejas que vienen dentro de una reseña en general positiva, y salen más casos que con reseñas enteras.
oraciones["texto_nlp"] = [" ".join(t) for t in con_bigramas(tokenizar(oraciones["oracion"].tolist()))]
# términos que no sirven para decidir nada (verbos genéricos, "lástima", "turista"...)
ALERT_STOP = {"turista", "lastima", "apto", "dejar", "tocar", "regresar", "persona", "gente", "nino", "pena", "vez", "pesar"}

def calcular_alertas(nivel, min_apar, min_neg):
    cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, min_df=3)
    Bm = cv.fit_transform(oraciones["texto_nlp"]).tocsc(); voc_b = np.array(cv.get_feature_names_out())
    filas = []
    for grupo in oraciones[nivel].dropna().unique():
        en = (oraciones[nivel] == grupo).values; neg = en & oraciones["es_negativa"].values; res = en & ~oraciones["es_negativa"].values
        if neg.sum() < min_neg: print(f"  {grupo}: solo {neg.sum()} oraciones negativas, las alertas no son confiables"); continue
        f_neg = np.asarray(Bm[neg].sum(axis=0)).ravel(); f_res = np.asarray(Bm[res].sum(axis=0)).ravel()
        lift = ((f_neg + 1) / (neg.sum() + 2)) / ((f_res + 1) / (res.sum() + 2))   # el +1 y +2 evitan dividir por cero
        cand = [j for j in np.where(f_neg >= min_apar)[0] if voc_b[j] not in ALERT_STOP]
        for j in sorted(cand, key=lambda j: -lift[j])[:15]:
            idx = np.where(neg & (Bm[:, j].toarray().ravel() > 0))[0]
            ej = oraciones.iloc[idx].sort_values("score").iloc[0]["oracion"][:180]     # la oración más negativa donde aparece, para interpretarla
            filas.append({nivel: grupo, "termino": voc_b[j], "n_oraciones_neg": int(f_neg[j]), "pct_oraciones_neg": round(f_neg[j] / neg.sum(), 3),
                          "lift": round(lift[j], 2), "ejemplo": ej})
    return pd.DataFrame(filas)

alertas = calcular_alertas("provincia", CONFIG["ALERTA_MIN_APARICIONES"], CONFIG["ALERTA_MIN_NEG"])
alertas_mun = calcular_alertas("municipio_nombre", CONFIG["ALERTA_MIN_APARICIONES_MUN"], CONFIG["ALERTA_MIN_NEG"])
alertas.to_csv(OUT / "alertas_negativas_provincia.csv", index=False); alertas_mun.to_csv(OUT / "alertas_negativas_municipio.csv", index=False)
print(f"Oraciones negativas: {int(oraciones['es_negativa'].sum()):,} de {len(oraciones):,}")
if len(alertas): display(alertas.groupby("provincia").head(10)[["provincia", "termino", "n_oraciones_neg", "lift", "ejemplo"]])

In [ ]:
from wordcloud import WordCloud
n = len(docs_mun); cols = 3; rows = int(np.ceil(n / cols))
fig, axs = plt.subplots(rows, cols, figsize=(14, 3.6 * rows)); axs = np.atleast_1d(axs).ravel()
for ax, mun in zip(axs, docs_mun.index):
    f = tfidf_terminos[tfidf_terminos["municipio"] == mun].set_index("termino")["tfidf"].to_dict()
    if f: ax.imshow(WordCloud(width=600, height=320, background_color="white", random_state=SEED).generate_from_frequencies(f))
    ax.set_title(mun); ax.axis("off")
for ax in axs[n:]: ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Tópicos con LDA
LDA trabaja con **conteos de palabras**, no con TF-IDF; por eso usamos `gensim` con `Dictionary` y `doc2bow`.
El número de tópicos **k** lo elegimos por **coherencia c_v** (se puede forzar con `LDA_K_FIJO`).

Ojo con la lectura: con reseñas tan cortas la coherencia queda baja (alrededor de 0,3 a 0,4) y los tópicos salen mezclados con los lugares (la mina y la catedral dominan Zipaquirá y Nemocón, el zoológico a San Antonio). Por eso los tratamos como **descripción de qué se comenta**, no como diagnóstico de problemas; para eso está la parte 6b.

In [ ]:
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
usable = resenas["n_tokens"] >= CONFIG["LDA_MIN_TOKENS"]
textos_lda = resenas.loc[usable, "tokens"].tolist(); idx_lda = resenas.index[usable]
dic = corpora.Dictionary(textos_lda); dic.filter_extremes(no_below=5, no_above=0.5)
corpus = [dic.doc2bow(t) for t in textos_lda]
keep = [i for i, b in enumerate(corpus) if len(b) > 0]
corpus = [corpus[i] for i in keep]; textos_lda = [textos_lda[i] for i in keep]; idx_lda = idx_lda[keep]
print(f"Documentos para LDA: {len(corpus):,} | vocabulario: {len(dic):,}")
assert len(corpus) >= 30, "Muy pocos documentos para LDA, necesitamos más reseñas."

def entrenar_lda(k):
    return LdaModel(corpus=corpus, id2word=dic, num_topics=k, passes=CONFIG["LDA_PASSES"], iterations=100,
                    alpha="auto", eta="auto", random_state=SEED, chunksize=2000, eval_every=None)

kmax = min(CONFIG["LDA_K_MAX"], max(CONFIG["LDA_K_MIN"], len(corpus) // 40))
if CONFIG["LDA_K_FIJO"]:
    K = CONFIG["LDA_K_FIJO"]; modelos = {K: entrenar_lda(K)}; coh = {}
else:
    modelos, coh = {}, {}
    for k in range(CONFIG["LDA_K_MIN"], kmax + 1):
        modelos[k] = entrenar_lda(k)
        coh[k] = CoherenceModel(model=modelos[k], texts=textos_lda, dictionary=dic, coherence="c_v", processes=1).get_coherence()
        print(f"  k={k}: coherencia c_v = {coh[k]:.3f}")
    K = max(coh, key=coh.get)
    plt.figure(figsize=(6, 3)); plt.plot(list(coh), list(coh.values()), "o-"); plt.axvline(K, c="r", ls="--")
    plt.xlabel("k"); plt.ylabel("c_v"); plt.title("Coherencia por número de tópicos"); plt.show()
if not CONFIG["LDA_K_FIJO"] and K in (CONFIG["LDA_K_MIN"], kmax):
    print(f"Ojo: el mejor k ({K}) quedó en el borde del rango; conviene ampliar LDA_K_MIN o LDA_K_MAX.")
lda = modelos[K]; print(f"\nk elegido = {K}. Si los tópicos no se entienden, probamos otro k con LDA_K_FIJO.")

In [ ]:
# Palabras de cada tópico y su etiqueta, todo automático.
# Etiqueta: si las palabras principales coinciden claramente con uno de los temas de SEMILLAS (vías, precios, atención...),
# usamos ese nombre; si no, nombramos el tópico con sus 3 palabras más características.
# "Más características" = palabras frecuentes en el tópico y a la vez poco frecuentes en el resto del corpus (relevancia, lambda=0.6).
SEMILLAS = {
 "Vías y acceso": ["via","carretera","trafico","peaje","acceso","camino","senalizacion","transito","trancon","hueco","destapar"],
 "Precios y costos": ["precio","caro","costo","cobrar","tarifa","pagar","valor","economico","costoso","boleta"],
 "Atención y servicio": ["atencion","servicio","personal","amable","atento","trato","grosero","atender"],
 "Parqueadero y filas": ["parqueadero","parquear","estacionamiento","fila","espera","demora","tiquete","esperar","demorar"],
 "Seguridad y limpieza": ["seguridad","limpio","limpieza","seguro","basura","vigilancia","sucio"],
 "Paisaje y naturaleza": ["paisaje","naturaleza","cascada","mirador","clima","tranquilo","verde","rio","aire","vista","embalse"],
 "Patrimonio y cultura": ["catedral","sal","mina","historia","cultura","museo","iglesia","historico","patrimonio","plaza"],
 "Alojamiento y gastronomía": ["comida","restaurante","hotel","cabana","finca","habitacion","desayuno","alojamiento","plato"],
}
p_corpus = np.array([dic.cfs.get(i, 1) for i in range(len(dic))], float); p_corpus /= p_corpus.sum()
tt = lda.get_topics()                                           # matriz tópicos x palabras
relevancia = 0.6 * np.log(tt + 1e-12) + 0.4 * np.log((tt + 1e-12) / p_corpus)

filas, etiquetas_auto, usados = [], {}, {}
for t in range(K):
    pals = [(dic[i], float(p)) for i, p in lda.get_topic_terms(t, topn=15)]
    masa = sum(p for _, p in pals)
    punt = {tema: sum(p for w, p in pals if set(w.split("_")) & set(sem)) / masa for tema, sem in SEMILLAS.items()}
    mejor = max(punt, key=punt.get)
    top_palabras = [dic[i].replace("_", " ") for i in np.argsort(relevancia[t])[::-1][:3]]
    if punt[mejor] >= 0.30: base = f"{mejor}: {', '.join(top_palabras[:2])}"     # nombre del tema + sus palabras más características
    else: base = " / ".join(top_palabras)
    usados[base] = usados.get(base, 0) + 1
    etiquetas_auto[t] = base if usados[base] == 1 else f"{base} ({usados[base]})"   # si se repite, distinguimos con un número
    filas += [dict(topico=t, rank=r + 1, palabra=w, peso=round(p, 4)) for r, (w, p) in enumerate(pals)]
ETIQUETAS_MANUAL = {}   # si una etiqueta automática no convence, la ponemos a mano. Ejemplo: {0: "Visita guiada a la mina"}
ETIQUETAS = {**etiquetas_auto, **ETIQUETAS_MANUAL}
topicos_palabras = pd.DataFrame(filas)
topicos_palabras["etiqueta"] = topicos_palabras["topico"].map(ETIQUETAS)
resumen = topicos_palabras.groupby("topico").apply(lambda g: ", ".join(g.sort_values("rank")["palabra"].head(10))).to_frame("palabras principales")
resumen["etiqueta"] = pd.Series(ETIQUETAS); display(resumen)
topicos_palabras.to_csv(OUT / "topicos_palabras.csv", index=False)

In [ ]:
# Distribución de tópicos por reseña, municipio y provincia
theta = np.zeros((len(corpus), K))
for i, b in enumerate(corpus):
    for t, p in lda.get_document_topics(b, minimum_probability=0): theta[i, t] = p
cols_t = [f"topico_{t}" for t in range(K)]
th = pd.DataFrame(theta, columns=cols_t, index=idx_lda)
resenas = resenas.drop(columns=[c for c in cols_t + ["topico_dominante", "prob_topico_dominante", "etiqueta_topico"] if c in resenas.columns]).join(th)
con_t = resenas[cols_t].notna().all(axis=1)   # las reseñas muy cortas quedan sin tópico, no se lo forzamos
resenas["topico_dominante"] = np.nan; resenas["prob_topico_dominante"] = np.nan
resenas.loc[con_t, "topico_dominante"] = resenas.loc[con_t, cols_t].idxmax(axis=1).str.replace("topico_", "").astype(float)
resenas.loc[con_t, "prob_topico_dominante"] = resenas.loc[con_t, cols_t].max(axis=1)
resenas["etiqueta_topico"] = resenas["topico_dominante"].map(ETIQUETAS)
print(f"Reseñas con tópico: {con_t.sum():,} de {len(resenas):,} (el resto tiene menos de {CONFIG['LDA_MIN_TOKENS']} tokens útiles)")

def ratio(grupo):
    d = resenas.dropna(subset=["topico_dominante"]).groupby(grupo)[cols_t].mean()
    d.columns = [ETIQUETAS[int(c.split("_")[1])] for c in cols_t]; return d
ratio_mun = ratio("municipio_nombre"); ratio_prov = ratio("provincia")
display(ratio_prov.round(3))
import seaborn as sns
plt.figure(figsize=(1.2 * K + 4, 3.6)); sns.heatmap(ratio_mun, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Proporción media de cada tópico por municipio"); plt.tight_layout(); plt.show()

# Qué tópicos se asocian con opiniones negativas (esto alimenta las reglas de decisión del proyecto)
d = resenas.dropna(subset=["topico_dominante"])
ts = pd.Series({ETIQUETAS[t]: np.average(d["sentimiento"], weights=d[f"topico_{t}"]) for t in range(K)}).sort_values()
plt.figure(figsize=(7, 0.45 * K + 1)); ts.plot(kind="barh", color=["#d9534f" if v < 0 else "#5cb85c" for v in ts]); plt.axvline(0, c="k", lw=.8)
plt.title("Sentimiento medio asociado a cada tópico"); plt.tight_layout(); plt.show()

# Tópicos dentro de las reseñas con alguna oración negativa, por provincia
neg_d = d[d["tiene_critica"]]
if len(neg_d): display(neg_d.groupby("provincia")[cols_t].mean().rename(columns={f"topico_{t}": ETIQUETAS[t] for t in range(K)}).round(3))
ratio_mun.to_csv(OUT / "topicos_por_municipio.csv"); ratio_prov.to_csv(OUT / "topicos_por_provincia.csv")

## 6b. Temas de problema y reglas de decisión
Los tópicos de LDA salen casi todos de elogios y descripciones del lugar, porque eso es lo que más escriben los visitantes. Para las **decisiones** nos interesa lo contrario: de qué se quejan. Por eso hacemos un análisis aparte, simple y transparente:
1. Tomamos solo las **oraciones negativas** (parte 4).
2. Les asignamos uno o varios **temas de problema** con diccionarios de palabras (vías, precios, atención, parqueadero, limpieza, seguridad...). Una oración puede tener varios temas; si no encaja en ninguno queda como `Sin tema identificado`.
3. Medimos qué **porcentaje de las reseñas** de cada municipio y provincia tiene una queja de cada tema, con intervalo de Wilson al 95 %.
4. Comparamos las dos provincias con la prueba exacta de Fisher (con corrección de Bonferroni por tener varios temas) y contrastamos las reglas de decisión del pitch con lo que salió, sin forzar el resultado.

Los diccionarios son editables (`TEMAS_PROBLEMA`) y el archivo `oraciones_negativas.csv` guarda cada oración con sus temas para revisar a mano que la asignación tenga sentido.

In [ ]:
from scipy.stats import fisher_exact

# Cada tema es una expresión regular sobre el texto sin tildes y en minúsculas
TEMAS_PROBLEMA = {
 "Vías y acceso": r"\b(vias?|carreteras?|trancon\w*|trafico|peajes?|huecos?|derrumbes?|senaliza\w*|transporte|buses|ruta|trocha|destapad\w*|pavimento|acceso|sin (?:carro|vehiculo|moto))\b",
 "Precios y cobros": r"\b(caro|cara|caros|caras|costos?|costos[oa]s?|cobr\w+|precios?|tarifas?|pagar|pagan|pago|plata|estafa\w*|sobrecosto\w*|exagerad\w*|abusiv\w*|dinero)\b",
 "Atención y servicio": r"\b(atencion|atender|atendi\w*|servicio|grosero|grosera|groseros|maltrato|mal trato|trato|personal|amabilidad|antipatic\w*|informacion|organizacion|desorganiz\w*|desorden|administracion|vendedores)\b",
 "Parqueadero, filas y aglomeración": r"\b(parqueaderos?|parquear|parqueo|estacionamiento|filas?|colas?|espera|esperar|demora\w*|demorar\w*|tiquetes?|aglomera\w*|congestion\w*|lleno|llena|sobrecupo|multitud)\b",
 "Limpieza y mantenimiento": r"\b(basura\w*|sucio|sucia|suciedad|limpieza|descuid\w*|abandon\w*|mantenimiento|olor|huele|oler|olores|deteriorad\w*|mal estado|danad\w*|banos?|sanitarios?|polvo|barro|mugre|desaseo|estiercol)\b",
 "Seguridad": r"\b(seguridad|insegur\w*|peligro\w*|robos?|atraco\w*|ladrones?|vigilancia|accidentes?|riesgo\w*|borrach\w*|drogas?)\b",
 "Infraestructura y comodidad": r"\b(sombra|sentarse|bancas?|sillas?|techo|iluminacion|carpas?|accesib\w*|discapacit\w*|movilidad reducida|adultos? mayor\w*|escaleras?|empinad\w*|resbal\w*|infraestructura|instalaciones)\b",
 "Horarios y restricciones": r"\b(cerrad[oa]s?|cerro|cierran|cierra|horarios?|no abre\w*|no dejan|no permit\w*|prohib\w*|no se puede|no pudimos|reservas?|cupos?|agotad\w*)\b",
 "Comida y gastronomía": r"\b(comida|restaurantes?|almuerzos?|platos?|sabor|crudo|desayunos?|bebidas?|menu)\b",
 "Ruido y ambiente": r"\b(ruido\w*|bares?|volumen|musica|borrach\w*|cantinas?|billares?)\b",
 "Obras y diseño del espacio": r"\b(remodel\w*|obras?|concreto|cemento|construccion|construyeron|diseno|mole)\b",
 "Bienestar animal": r"\b(jaulas?|encierros?|enjaul\w*|cautiv\w*|maltrato animal)\b",
}
# Qué acción institucional sugiere cada tema (solo se muestra si la diferencia entre provincias es significativa)
ACCION = {
 "Vías y acceso": "priorizar inversión vial, señalización y transporte de acceso",
 "Precios y cobros": "revisar y regular tarifas; informar precios de forma visible",
 "Atención y servicio": "capacitación en servicio al cliente con prestadores del RNT",
 "Parqueadero, filas y aglomeración": "ordenar parqueo, filas y control de aforo",
 "Limpieza y mantenimiento": "reforzar manejo de residuos, baños y mantenimiento de espacios públicos",
 "Seguridad": "reforzar vigilancia y seguridad en los atractivos",
 "Infraestructura y comodidad": "mejorar sombra, asientos, accesibilidad y señalización interna",
 "Horarios y restricciones": "informar horarios y restricciones antes de la visita",
 "Comida y gastronomía": "acompañar a restaurantes y comerciantes en calidad y precios",
 "Ruido y ambiente": "regular ruido y usos del espacio público",
 "Obras y diseño del espacio": "revisar diseño y obras en parques y plazas",
 "Bienestar animal": "revisar condiciones de los animales con la autoridad ambiental",
}
PAT = {k: re.compile(v) for k, v in TEMAS_PROBLEMA.items()}
SIN_TEMA = "Sin tema identificado"
def temas_de(t): return [k for k, p in PAT.items() if p.search(unidecode(t.lower()))] or [SIN_TEMA]

neg_or = oraciones[oraciones["es_negativa"]].copy()
neg_or["temas"] = neg_or["oracion"].map(temas_de)
neg_or[["id_resena", "provincia", "municipio_nombre", "lugar_nombre", "oracion", "p_neg"]].assign(temas=neg_or["temas"].map(" | ".join)) \
      .round({"p_neg": 3}).to_csv(OUT / "oraciones_negativas.csv", index=False)
ex = neg_or.explode("temas").rename(columns={"temas": "tema"})
print(f"Oraciones negativas: {len(neg_or):,} | sin tema identificado: {(neg_or['temas'].map(lambda x: x == [SIN_TEMA])).mean():.0%}")

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); a_ = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - a_) / d, (c + a_) / d)

TEMAS_LISTA = list(TEMAS_PROBLEMA) + [SIN_TEMA]
filas = []
for nivel, col in [("provincia", "provincia"), ("municipio", "municipio_nombre")]:
    for grupo, n_grupo in resenas[col].value_counts().items():
        sub = ex[ex[col] == grupo]
        for tema in TEMAS_LISTA:
            s = sub[sub["tema"] == tema]; k = s["id_resena"].nunique(); lo, hi = wilson(k, n_grupo)
            filas.append(dict(nivel=nivel, grupo=grupo, tema=tema, n_resenas_tema=k, n_oraciones=len(s), n_resenas_grupo=int(n_grupo),
                              pct_resenas=round(k / n_grupo, 4), ic95_inf=round(lo, 4), ic95_sup=round(hi, 4)))
problemas = pd.DataFrame(filas); problemas.to_csv(OUT / "problemas_por_tema.csv", index=False)
print("\nPorcentaje de reseñas con una queja por tema, por provincia (IC 95%):")
pv = problemas[(problemas["nivel"] == "provincia") & (problemas["tema"] != SIN_TEMA)]
display(pv.pivot(index="tema", columns="grupo", values="pct_resenas").mul(100).round(1).assign(n_total=pv.groupby("tema")["n_resenas_tema"].sum()).sort_values("n_total", ascending=False))

# Comparación entre provincias (Fisher exacto, Bonferroni por número de temas)
PA, PB = "Sabana Centro", "Tequendama"
nA, nB = int((resenas["provincia"] == PA).sum()), int((resenas["provincia"] == PB).sum())
filas = []
for tema in TEMAS_PROBLEMA:
    kA = int(pv[(pv["grupo"] == PA) & (pv["tema"] == tema)]["n_resenas_tema"].iloc[0]); kB = int(pv[(pv["grupo"] == PB) & (pv["tema"] == tema)]["n_resenas_tema"].iloc[0])
    p = fisher_exact([[kA, nA - kA], [kB, nB - kB]])[1]; pa_ = min(1.0, p * len(TEMAS_PROBLEMA))
    mayor = PA if kA / nA > kB / nB else PB
    sig = pa_ < 0.05
    filas.append(dict(tema=tema, pct_sabana_centro=round(kA / nA, 4), pct_tequendama=round(kB / nB, 4), diferencia_pp=round((kA / nA - kB / nB) * 100, 2),
                      n_sabana_centro=kA, n_tequendama=kB, p_valor=round(p, 4), p_ajustada=round(pa_, 4), significativa=bool(sig), provincia_mayor=mayor,
                      solidez="suficiente" if kA + kB >= 15 else "baja (pocas reseñas)",
                      accion_sugerida=(f"En {mayor}: {ACCION[tema]}" if sig else "Sin diferencia significativa entre provincias")))
reglas = pd.DataFrame(filas).sort_values("p_valor"); reglas.to_csv(OUT / "reglas_decision.csv", index=False)
display(reglas.drop(columns=["accion_sugerida"]))

# ¿Se cumplen las reglas que planteamos en el pitch? Lo decimos tal como salga.
def evaluar_regla(texto, tema, provincia_esperada):
    r = reglas[reglas["tema"] == tema].iloc[0]
    ok = bool(r["significativa"]) and r["provincia_mayor"] == provincia_esperada
    print(f"- {texto}: {tema} aparece en {r['pct_sabana_centro']:.1%} de las reseñas de Sabana Centro y en {r['pct_tequendama']:.1%} de Tequendama "
          f"(p ajustada = {r['p_ajustada']:.3f}) -> {'la regla SÍ se respalda' if ok else 'la regla NO se respalda con estos datos'}")
print("\nContraste con las reglas del pitch:")
evaluar_regla("Regla Tequendama (priorizar vías y señalización)", "Vías y acceso", PB)
evaluar_regla("Regla Sabana Centro (talleres de servicio)", "Atención y servicio", PA)
evaluar_regla("Regla Sabana Centro (regular tarifas)", "Precios y cobros", PA)

fig, ax = plt.subplots(figsize=(8, 0.5 * len(TEMAS_PROBLEMA) + 1.5))
tab = pv.pivot(index="tema", columns="grupo", values="pct_resenas").mul(100); tab = tab.loc[tab.sum(axis=1).sort_values().index]
tab.plot(kind="barh", ax=ax, color=["#3b7dd8", "#7a9e7e"]); ax.set_xlabel("% de reseñas con una queja del tema"); ax.set_ylabel(""); ax.set_title("Temas de problema por provincia")
plt.tight_layout(); plt.show()

## 7. Cruce con el RNT
El RNT solo permite comparar por municipio, o sea 6 puntos. Con n = 6 la correlación es **solo descriptiva**: sirve para contextualizar, no para decir que una cosa causa la otra.

In [ ]:
from scipy.stats import spearmanr
if resumen_rnt is None:
    print("No hay RNT; guardamos solo las métricas de sentimiento.")
    cruce = met_mun.copy()
else:
    cruce = (met_mun.merge(resumen_rnt, on=["provincia", "municipio_nombre"], how="left")
                    .assign(resenas_por_prestador=lambda d: d["n_resenas"] / d["prestadores_rnt"]))
    display(cruce[["provincia","municipio_nombre","prestadores_rnt","n_resenas","resenas_por_prestador","sent_medio"]].round(3))
    if cruce[["prestadores_rnt", "sent_medio"]].dropna().shape[0] >= 4:
        rho, p = spearmanr(cruce["prestadores_rnt"], cruce["sent_medio"], nan_policy="omit")
        print(f"Spearman (prestadores RNT vs sentimiento medio), n={len(cruce)}: rho={rho:.2f}, p={p:.2f}. Es descriptivo, n es muy chico.")
    sc = cruce.dropna(subset=["prestadores_rnt"])
    plt.figure(figsize=(6, 4)); plt.scatter(sc["prestadores_rnt"], sc["sent_medio"], s=80)
    for _, r in sc.iterrows(): plt.annotate(r["municipio_nombre"], (r["prestadores_rnt"], r["sent_medio"]), fontsize=8, xytext=(5, 4), textcoords="offset points")
    plt.xlabel("Prestadores activos en el RNT"); plt.ylabel("Sentimiento medio"); plt.title("RNT vs percepción (descriptivo, n=6)")
    plt.tight_layout(); plt.show()
cruce.to_csv(OUT / "metricas_municipio.csv", index=False); met_prov.to_csv(OUT / "metricas_provincia.csv", index=False)

## 8. Exportar resultados y modelos

In [ ]:
cols_export = ["id_resena","provincia","municipio_nombre","atractivo_turistico","lugar_nombre","fuente","fecha","truncada","texto_resena","texto_nlp",
               "p_pos","p_neg","p_neu","sentimiento","sent_label","sent_min","n_oraciones","n_oraciones_neg","tiene_critica","mixta","sent_metodo",
               "topico_dominante","etiqueta_topico","prob_topico_dominante"] + cols_t
resenas[[c for c in cols_export if c in resenas.columns]].to_csv(OUT / "resenas_procesadas.csv", index=False)
lda.save(str(OUT / "lda_model.gensim")); dic.save(str(OUT / "lda_dictionary.dict"))

import sklearn, gensim
config_run = dict(fecha_ejecucion=str(pd.Timestamp.now()), n_resenas=int(len(resenas)), n_oraciones=int(len(oraciones)), k_lda=int(K),
                  coherencia_cv={int(k): round(float(v), 3) for k, v in coh.items()},
                  metodo_sentimiento=str(resenas["sent_metodo"].iloc[0]), etiquetas_topicos={int(k): v for k, v in ETIQUETAS.items()},
                  config={k: (list(v) if isinstance(v, set) else v) for k, v in CONFIG.items()},
                  versiones=dict(python=platform.python_version(), pandas=pd.__version__, spacy=spacy.__version__, sklearn=sklearn.__version__, gensim=gensim.__version__))
(OUT / "config_ejecucion.json").write_text(json.dumps(config_run, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# La caché vieja (versión por reseña completa) ya no sirve; si quedó en la carpeta, la quitamos para no confundir
(OUT / "cache_sentimiento.csv").unlink(missing_ok=True)

zip_path = Path(str(OUT) + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"): z.write(f, f.relative_to(OUT.parent))
print("Archivos generados en salidas/:")
for p in sorted(OUT.iterdir()): print("  ", p.name, f"({p.stat().st_size/1024:.0f} KB)")
if EN_COLAB:
    from google.colab import files; files.download(str(zip_path))